In [3]:
# Task 1: Fetch data from HackerNews API and save to JSON

import requests
import time
import json
import os
from datetime import datetime

# Base URLs for HackerNews API
TOP_STORIES_URL = "https://hacker-news.firebaseio.com/v0/topstories.json"
ITEM_URL = "https://hacker-news.firebaseio.com/v0/item/{}.json"

# Categories and keywords for classification
categories = {
    "technology": ["ai", "software", "tech", "code", "computer", "data", "cloud", "api", "gpu", "llm"],
    "worldnews": ["war", "government", "country", "president", "election", "climate", "attack", "global"],
    "sports": ["nfl", "nba", "fifa", "sport", "game", "team", "player", "league", "championship"],
    "science": ["research", "study", "space", "physics", "biology", "discovery", "nasa", "genome"],
    "entertainment": ["movie", "film", "music", "netflix", "game", "book", "show", "award", "streaming"]
}

# Function to assign category based on title keywords
def assign_category(title):
    title = title.lower()
    for category, keywords in categories.items():
        for word in keywords:
            if word in title:
                return category
    return None  # Ignore if no category matches

# Fetch top story IDs
headers = {"User-Agent": "TrendPulse/1.0"}
story_ids = requests.get(TOP_STORIES_URL, headers=headers).json()

stories = []
category_count = {cat: 0 for cat in categories}

# Fetch more stories to ensure enough valid ones
for story_id in story_ids[:800]:   # increased limit

    response = requests.get(ITEM_URL.format(story_id), headers=headers)

    if response.status_code != 200:
        continue

    data = response.json()
    if not data or "title" not in data:
        continue

    # Assign category
    category = assign_category(data["title"])

    # If no category match → assign "other" instead of skipping
    if not category:
        category = "other"

    # Limit only main categories (not "other")
    if category in category_count:
        if category_count[category] >= 25:
            continue
        category_count[category] += 1

    story = {
        "post_id": data.get("id"),
        "title": data.get("title"),
        "category": category,
        "score": data.get("score", 0),
        "num_comments": data.get("descendants", 0),
        "author": data.get("by", "unknown"),
        "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

    stories.append(story)

    # Stop when we reach at least 120+
    if len(stories) >= 125:
        break

# Create data folder if not exists
os.makedirs("data", exist_ok=True)

# Save JSON file with current date
filename = f"data/trends_{datetime.now().strftime('%Y%m%d')}.json"

with open(filename, "w") as f:
    json.dump(stories, f, indent=4)

# Print result
print(f"Collected {len(stories)} stories. Saved to {filename}")

Collected 125 stories. Saved to data/trends_20260414.json
